In [2]:
from functions.utils.feature_selection.boruta_selection import boruta_selection
from functions.utils.feature_selection.markov_boundary_selection_dcor import markov_boundary_selection_dcor
import numpy as np

# Classification

## 1.Linear Regression

In [3]:
rng = np.random.default_rng(0)
n = 500

x = rng.normal(size=(n,6))
fx = 3 * x[:, 0] - 2 * x[:, 2] + 1.5 * x[:, 4] + rng.normal(size=n)
probs = 1 / (1 + np.exp(-fx))
y = (probs > 0.5).astype(int)


In [4]:
boruta_selected_features = boruta_selection(x=x,y=y)
boruta_selected_features

[0, 2, 4]

In [5]:
mb_selected_features = markov_boundary_selection_dcor(x=x, y=y, alpha=0.05)
mb_selected_features

{0: 28.0, 2: 35.0, 4: 37.0}

## 2. Non-Linear Regression

### 2-1. Polynomial Regression

In [6]:
import numpy as np

# Set the random number generator for reproducibility
rng = np.random.default_rng(0)
n = 500  # Number of samples

# Create the correlation matrix (e.g., correlations between features)
corr_matrix = np.eye(6)  # Start with identity matrix (no correlation)

# Define some correlations between features
corr_matrix[0, 1] = 0.8  # Strong correlation between feature 0 and feature 1
corr_matrix[1, 0] = 0.8  # Symmetric correlation
corr_matrix[2, 3] = 0.5  # Moderate correlation between feature 2 and feature 3
corr_matrix[3, 2] = 0.5  # Symmetric correlation
corr_matrix[4, 5] = 0.3  # Weak correlation between feature 4 and feature 5
corr_matrix[5, 4] = 0.3  # Symmetric correlation

# Perform Cholesky decomposition to generate correlated features
chol_decomp = np.linalg.cholesky(corr_matrix)

# Generate uncorrelated random data
x_uncorr = rng.normal(size=(n, 6))

# Create correlated data by multiplying the uncorrelated data by the Cholesky decomposition
x = x_uncorr @ chol_decomp.T  # @ is matrix multiplication

# Create a non-linear relationship for y (quadratic and cubic terms)
fx = (
    3 * x[:, 0] ** 2            # quadratic term
    - 2 * x[:, 1] ** 3          # cubic term
    + 1.5 * x[:, 2]             # linear term
    + rng.normal(size=n)        # noise
)

# Apply logistic transformation to get probabilities
probs = 1 / (1 + np.exp(-fx))

# Convert probabilities to binary target variable
y = (probs > 0.5).astype(int)

# Now x is a matrix with correlated features, and y is the binary target variable


In [7]:
boruta_selected_features = boruta_selection(x=x,y=y)
boruta_selected_features

[0, 1, 2, 3]

In [8]:
mb_selected_features = markov_boundary_selection_dcor(x=x, y=y,alpha=0.05)
mb_selected_features

{0: 49.0, 1: 48.0, 2: 48.0, 3: 35.0}

### 2-2. Sin, Cos, Exp and Log

In [9]:
# Set random number generator for reproducibility
rng = np.random.default_rng(0)

# Number of samples and features
n = 500
n_features = 300

# Define the correlation matrix (here we use some example correlations)
corr_matrix = np.eye(n_features)  # Start with an identity matrix (no correlation)
corr_matrix[0, 1] = 0.8  # Strong correlation between feature 0 and feature 1
corr_matrix[1, 0] = 0.8  # Symmetric
corr_matrix[2, 3] = 0.5  # Moderate correlation between feature 2 and feature 3
corr_matrix[3, 2] = 0.5  # Symmetric
corr_matrix[4, 5] = 0.3  # Weak correlation between feature 4 and feature 5
corr_matrix[5, 4] = 0.3  # Symmetric

# Generate correlated features based on the correlation matrix
# Use a Cholesky decomposition to obtain a lower triangular matrix
chol_decomp = np.linalg.cholesky(corr_matrix)

# Generate uncorrelated random data
x_uncorr = rng.normal(size=(n, n_features))

# Create correlated data by multiplying the uncorrelated data by the Cholesky decomposition
x = x_uncorr @ chol_decomp.T  # @ is matrix multiplication

# Now you have a feature matrix `x` where the features have the specified correlations

# Define the target variable (same as in your example)
fx = (
    3 * x[:, 0] ** 2            # quadratic term
    - 2 * x[:, 1] ** 3          # cubic term
    + 1.5 * np.sin(x[:, 2])     # sine term
    + 2 * np.cos(x[:, 3])      # cosine term
    + 0.5 * np.exp(x[:, 4])     # exponential term
    + 0.5 * np.log(np.abs(x[:, 5]) + 1)  # log term (handling x > 0 for log)
    + rng.normal(size=n)       # noise
)

# Apply logistic transformation to get probabilities
probs = 1 / (1 + np.exp(-fx))

# Convert probabilities to binary target variable
y = (probs > 0.5).astype(int)

# Now `x` has the specified correlation structure, and `y` is the binary target variable


In [10]:
boruta_selected_features = boruta_selection(x=x, y=y)
boruta_selected_features

[0, 1, 2, 3, 12, 18, 72, 161]

In [11]:
mb_selected_features = markov_boundary_selection_dcor(x=x, y=y, alpha=0.05, n_resamples=500)
mb_selected_features

{0: 26.0, 1: 24.0, 2: 19.0, 3: 30.0, 4: 24.0, 98: 20.0, 187: 26.0, 200: 21.0}